# Final Model Interpretation

This notebook synthesizes the results from the machine-learning experiments performed on the security-related code review comment dataset.

The analysis combines:

- 80/20 held-out test-set model comparison
- 5-fold stratified cross-validation
- Confusion-matrix and error analysis
- Feature-level analysis
- Security-category analysis
- Cross-model disagreement analysis

The purpose is to interpret the observed model behavior, identify common strengths and weaknesses, and document limitations of the constructed dataset and labeling process.

The analysis focuses on Logistic Regression, Linear SVM, and Random Forest models using TF-IDF features.

In [1]:
import os
import pandas as pd
import numpy as np

from IPython.display import display

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA_DIR = os.path.join(
    PROJECT_ROOT,
    "data"
)

PROCESSED_DIR = os.path.join(
    DATA_DIR,
    "processed"
)

RESULTS_DIR = os.path.join(
    DATA_DIR,
    "results"
)

print("Project root:")
print(PROJECT_ROOT)

print("\nResults directory:")
print(RESULTS_DIR)

Project root:
/Users/mustafaahmed/Documents/Security Code Review Analysis

Results directory:
/Users/mustafaahmed/Documents/Security Code Review Analysis/data/results


In [3]:
FINAL_DATA_PATH = os.path.join(
    PROCESSED_DIR,
    "final_security.csv"
)

df = pd.read_csv(FINAL_DATA_PATH)

print("Final dataset shape:", df.shape)

print("\nLabel distribution:")
display(
    df["security_label"]
    .value_counts()
    .sort_index()
)

Final dataset shape: (9230, 13)

Label distribution:


security_label
0    6119
1    3028
2      83
Name: count, dtype: int64

In [4]:
binary_df = df[
    df["security_label"].isin([0, 1])
].copy()

binary_df["actual_label"] = (
    binary_df["security_label"]
    .astype(int)
)

print("Binary classification dataset:", binary_df.shape)

print("\nClass distribution:")
display(
    binary_df["actual_label"]
    .value_counts()
    .sort_index()
)

Binary classification dataset: (9147, 14)

Class distribution:


actual_label
0    6119
1    3028
Name: count, dtype: int64

In [5]:
CV_RESULTS_PATH = os.path.join(
    RESULTS_DIR,
    "cross_validation_results.csv"
)

CV_SUMMARY_PATH = os.path.join(
    RESULTS_DIR,
    "cross_validation_summary.csv"
)

cv_results = pd.read_csv(CV_RESULTS_PATH)
cv_summary = pd.read_csv(CV_SUMMARY_PATH)

print("Cross-validation results:")
display(cv_results)

print("\nCross-validation summary:")
display(cv_summary)

Cross-validation results:


,model,fold,accuracy,balanced_accuracy,precision,recall,f1_score,training_time_seconds,prediction_time_seconds
0,Logistic Regression,1,0.967760,0.955069,0.984071,0.917492,0.949616,0.131188,0.021841
1,Logistic Regression,2,0.965027,0.950111,0.987410,0.905941,0.944923,0.122964,0.020899
2,Logistic Regression,3,0.973756,0.962420,0.991182,0.928926,0.959044,0.120302,0.020374
3,Logistic Regression,4,0.960634,0.945093,0.980180,0.899174,0.937931,0.131586,0.022063
4,Logistic Regression,5,0.961181,0.945582,0.981982,0.899340,0.938846,0.126751,0.021672
5,Linear SVM,1,0.976503,0.967021,0.989565,0.938944,0.963590,0.115560,0.020609
6,Linear SVM,2,0.971038,0.958354,0.991119,0.920792,0.954662,0.113881,0.019925
7,Linear SVM,3,0.978677,0.970694,0.987931,0.947107,0.967089,0.113819,0.019491
8,Linear SVM,4,0.966648,0.952094,0.989209,0.909091,0.947459,0.127409,0.022375
9,Linear SVM,5,0.971022,0.959184,0.987654,0.924092,0.954817,0.126483,0.022738



Cross-validation summary:


,model,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,training_time_mean,prediction_time_mean
0,Linear SVM,0.972778,0.004805,0.961470,0.007395,0.989096,0.001393,0.928005,0.015080,0.957523,0.007828,0.119430,0.021028
1,Logistic Regression,0.965672,0.005378,0.951655,0.007244,0.984965,0.004394,0.910174,0.012860,0.946072,0.008670,0.126558,0.021370
2,Random Forest,0.963485,0.005166,0.946934,0.007438,0.990878,0.002945,0.897954,0.014258,0.942090,0.008542,0.866010,0.061732


In [6]:
cv_interpretation = cv_summary[
    [
        "model",
        "accuracy_mean",
        "accuracy_std",
        "balanced_accuracy_mean",
        "balanced_accuracy_std",
        "precision_mean",
        "precision_std",
        "recall_mean",
        "recall_std",
        "f1_mean",
        "f1_std"
    ]
].copy()

display(cv_interpretation)

,model,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
0,Linear SVM,0.972778,0.004805,0.961470,0.007395,0.989096,0.001393,0.928005,0.015080,0.957523,0.007828
1,Logistic Regression,0.965672,0.005378,0.951655,0.007244,0.984965,0.004394,0.910174,0.012860,0.946072,0.008670
2,Random Forest,0.963485,0.005166,0.946934,0.007438,0.990878,0.002945,0.897954,0.014258,0.942090,0.008542


In [7]:
MODEL_COMPARISON_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "model_comparison",
    "model_comparison_results.csv"
)

print("Looking for:")
print(MODEL_COMPARISON_PATH)

if os.path.exists(MODEL_COMPARISON_PATH):
    model_comparison_df = pd.read_csv(MODEL_COMPARISON_PATH)

    print("\nModel comparison results loaded successfully.")
    print("Shape:", model_comparison_df.shape)

    display(model_comparison_df)

else:
    print("\nERROR: File not found.")
    print("Expected path:")
    print(MODEL_COMPARISON_PATH)

Looking for:
/Users/mustafaahmed/Documents/Security Code Review Analysis/data/processed/model_comparison/model_comparison_results.csv

Model comparison results loaded successfully.
Shape: (3, 12)


,model,accuracy,balanced_accuracy,precision,recall,f1_score,true_negatives,false_positives,false_negatives,true_positives,training_time_seconds,prediction_time_seconds
0,Linear SVM,0.975956,0.964946,0.994718,0.932343,0.962521,1221,3,41,565,0.012058,0.000185
1,Logistic Regression,0.970492,0.956279,0.996403,0.914191,0.953528,1222,2,52,554,0.048118,0.000229
2,Random Forest,0.968306,0.952562,0.998182,0.905941,0.949827,1223,1,57,549,0.747655,0.035687


In [8]:
heldout_results = pd.DataFrame({
    "model": [
        "Linear SVM",
        "Logistic Regression",
        "Random Forest"
    ],
    "accuracy": [
        0.975956,
        0.970492,
        0.968306
    ],
    "balanced_accuracy": [
        0.964946,
        0.956279,
        0.952562
    ],
    "precision": [
        0.994718,
        0.996403,
        0.998182
    ],
    "recall": [
        0.932343,
        0.914191,
        0.905941
    ],
    "f1_score": [
        0.962521,
        0.953528,
        0.949827
    ],
    "true_negative": [
        1221,
        1222,
        1223
    ],
    "false_positive": [
        3,
        2,
        1
    ],
    "false_negative": [
        41,
        52,
        57
    ],
    "true_positive": [
        565,
        554,
        549
    ]
})

display(heldout_results)

,model,accuracy,balanced_accuracy,precision,recall,f1_score,true_negative,false_positive,false_negative,true_positive
0,Linear SVM,0.975956,0.964946,0.994718,0.932343,0.962521,1221,3,41,565
1,Logistic Regression,0.970492,0.956279,0.996403,0.914191,0.953528,1222,2,52,554
2,Random Forest,0.968306,0.952562,0.998182,0.905941,0.949827,1223,1,57,549


In [9]:
final_comparison = cv_summary[
    [
        "model",
        "accuracy_mean",
        "balanced_accuracy_mean",
        "precision_mean",
        "recall_mean",
        "f1_mean"
    ]
].copy()

final_comparison = final_comparison.rename(
    columns={
        "accuracy_mean": "cv_accuracy",
        "balanced_accuracy_mean": "cv_balanced_accuracy",
        "precision_mean": "cv_precision",
        "recall_mean": "cv_recall",
        "f1_mean": "cv_f1"
    }
)

final_comparison = final_comparison.merge(
    heldout_results[
        [
            "model",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "f1_score"
        ]
    ],
    on="model"
)

final_comparison = final_comparison[
    [
        "model",
        "accuracy",
        "cv_accuracy",
        "balanced_accuracy",
        "cv_balanced_accuracy",
        "precision",
        "cv_precision",
        "recall",
        "cv_recall",
        "f1_score",
        "cv_f1"
    ]
]

display(final_comparison)

,model,accuracy,cv_accuracy,balanced_accuracy,cv_balanced_accuracy,precision,cv_precision,recall,cv_recall,f1_score,cv_f1
0,Linear SVM,0.975956,0.972778,0.964946,0.961470,0.994718,0.989096,0.932343,0.928005,0.962521,0.957523
1,Logistic Regression,0.970492,0.965672,0.956279,0.951655,0.996403,0.984965,0.914191,0.910174,0.953528,0.946072
2,Random Forest,0.968306,0.963485,0.952562,0.946934,0.998182,0.990878,0.905941,0.897954,0.949827,0.942090


In [10]:
difference_df = pd.DataFrame({
    "model": heldout_results["model"],
    "accuracy_difference": (
        heldout_results["accuracy"].values
        -
        cv_summary["accuracy_mean"].values
    ),
    "balanced_accuracy_difference": (
        heldout_results["balanced_accuracy"].values
        -
        cv_summary["balanced_accuracy_mean"].values
    ),
    "precision_difference": (
        heldout_results["precision"].values
        -
        cv_summary["precision_mean"].values
    ),
    "recall_difference": (
        heldout_results["recall"].values
        -
        cv_summary["recall_mean"].values
    ),
    "f1_difference": (
        heldout_results["f1_score"].values
        -
        cv_summary["f1_mean"].values
    )
})

display(difference_df)

,model,accuracy_difference,balanced_accuracy_difference,precision_difference,recall_difference,f1_difference
0,Linear SVM,0.003178,0.003476,0.005622,0.004338,0.004998
1,Logistic Regression,0.004820,0.004624,0.011438,0.004017,0.007456
2,Random Forest,0.004821,0.005628,0.007304,0.007987,0.007737


In [11]:
error_profile = heldout_results[
    [
        "model",
        "false_positive",
        "false_negative"
    ]
].copy()

error_profile["total_errors"] = (
    error_profile["false_positive"]
    +
    error_profile["false_negative"]
)

display(error_profile)

,model,false_positive,false_negative,total_errors
0,Linear SVM,3,41,44
1,Logistic Regression,2,52,54
2,Random Forest,1,57,58


In [12]:
disagreement_summary = pd.DataFrame({
    "disagreement_type": [
        "All models agree",
        "LR + RF agree, SVM disagrees",
        "LR + SVM agree, RF disagrees",
        "SVM + RF agree, LR disagrees"
    ],
    "count": [
        1806,
        10,
        10,
        4
    ]
})

display(disagreement_summary)

total_predictions = disagreement_summary["count"].sum()

disagreement_summary["percentage"] = (
    disagreement_summary["count"]
    / total_predictions
    * 100
)

display(disagreement_summary)

,disagreement_type,count
0,All models agree,1806
1,"LR + RF agree, SVM disagrees",10
2,"LR + SVM agree, RF disagrees",10
3,"SVM + RF agree, LR disagrees",4


,disagreement_type,count,percentage
0,All models agree,1806,98.688525
1,"LR + RF agree, SVM disagrees",10,0.546448
2,"LR + SVM agree, RF disagrees",10,0.546448
3,"SVM + RF agree, LR disagrees",4,0.218579


In [13]:
category_results = pd.DataFrame({
    "category": [
        "authentication",
        "secrets",
        "authorization",
        "cryptography",
        "vulnerability",
        "general_security",
        "injection",
        "input_validation",
        "memory_safety",
        "malicious_activity",
        "code_execution"
    ],
    "sample_size": [
        585,
        560,
        438,
        405,
        368,
        256,
        142,
        88,
        32,
        11,
        8
    ],
    "f1_score": [
        0.6769,
        0.7470,
        0.5882,
        0.6512,
        0.9973,
        0.9574,
        1.0000,
        0.8824,
        1.0000,
        1.0000,
        0.9333
    ]
})

display(category_results)

,category,sample_size,f1_score
0,authentication,585,0.6769
1,secrets,560,0.7470
2,authorization,438,0.5882
3,cryptography,405,0.6512
4,vulnerability,368,0.9973
5,general_security,256,0.9574
6,injection,142,1.0000
7,input_validation,88,0.8824
8,memory_safety,32,1.0000
9,malicious_activity,11,1.0000


In [14]:
category_results[
    category_results["sample_size"] >= 100
].sort_values(
    "f1_score"
)

,category,sample_size,f1_score
2,authorization,438,0.5882
3,cryptography,405,0.6512
0,authentication,585,0.6769
1,secrets,560,0.7470
5,general_security,256,0.9574
4,vulnerability,368,0.9973
6,injection,142,1.0000


In [15]:
positive_features = pd.DataFrame({
    "feature": [
        "vulnerability",
        "csrf",
        "exploit",
        "vulnerable",
        "security issue",
        "xss",
        "vulnerabilities",
        "malicious",
        "unauthorized",
        "insecure",
        "cve",
        "injection",
        "attacker",
        "leak",
        "rce",
        "privilege escalation"
    ],
    "coefficient": [
        5.593,
        5.437,
        5.103,
        4.995,
        4.669,
        4.460,
        4.186,
        4.014,
        3.441,
        3.187,
        2.515,
        2.340,
        2.263,
        2.150,
        2.081,
        1.590
    ]
})

negative_features = pd.DataFrame({
    "feature": [
        "authorization",
        "tls",
        "credentials",
        "token",
        "permissions",
        "password",
        "secret",
        "certificate",
        "ssl",
        "authentication",
        "privileges",
        "session"
    ],
    "coefficient": [
        -4.211,
        -2.447,
        -2.188,
        -2.168,
        -2.142,
        -2.028,
        -1.959,
        -1.944,
        -1.804,
        -1.694,
        -1.644,
        -1.565
    ]
})

print("Features associated with the positive class:")
display(positive_features)

print("\nFeatures associated with the negative class:")
display(negative_features)

Features associated with the positive class:


,feature,coefficient
0,vulnerability,5.593
1,csrf,5.437
2,exploit,5.103
3,vulnerable,4.995
4,security issue,4.669
5,xss,4.460
6,vulnerabilities,4.186
7,malicious,4.014
8,unauthorized,3.441
9,insecure,3.187



Features associated with the negative class:


,feature,coefficient
0,authorization,-4.211
1,tls,-2.447
2,credentials,-2.188
3,token,-2.168
4,permissions,-2.142
5,password,-2.028
6,secret,-1.959
7,certificate,-1.944
8,ssl,-1.804
9,authentication,-1.694


## Feature-Level Interpretation

The Linear SVM feature coefficients indicate that explicit vulnerability terminology is strongly associated with the security-related class. Examples include terms such as `vulnerability`, `csrf`, `exploit`, `vulnerable`, `xss`, `malicious`, `unauthorized`, `insecure`, and `cve`.

In contrast, several general security terms, including `authorization`, `token`, `permissions`, `password`, `secret`, `authentication`, and `session`, have negative coefficients in the fitted model.

This does not mean that these terms are inherently non-security-related. Instead, the coefficients indicate that within this dataset, these terms frequently occur in comments labeled as non-security-related. This is consistent with the error analysis, which found that context-dependent authentication, authorization, token, and credential discussions were among the more difficult cases.

The feature analysis therefore suggests that explicit vulnerability terminology provides stronger classification signals than general security vocabulary alone.

# Main Findings

## 1. Overall model performance

All three evaluated models achieved strong classification performance on the constructed binary dataset. Across the 5-fold stratified cross-validation experiment, Linear SVM achieved the highest mean accuracy, balanced accuracy, recall, and F1 score, while Random Forest achieved the highest mean precision.

The cross-validation results were broadly consistent with the earlier 80/20 held-out test-set comparison. This consistency provides evidence that the observed performance differences were not limited to a single train-test partition.

## 2. Model stability

F1-score variation across the five folds was relatively small for all three models. Linear SVM had a mean F1 score of 0.9575 with a standard deviation of 0.0078, while Logistic Regression and Random Forest had mean F1 scores of 0.9461 and 0.9421 respectively.

The relatively small variation across folds indicates that model performance remained reasonably consistent across the different validation partitions.

## 3. False negatives remain the primary error type

On the held-out test set, all three models produced substantially more false negatives than false positives.

Linear SVM produced 3 false positives and 41 false negatives. Logistic Regression produced 2 false positives and 52 false negatives, while Random Forest produced 1 false positive and 57 false negatives.

This indicates that the main classification challenge is identifying security-related comments that use indirect or contextual language rather than producing large numbers of false security classifications.

## 4. Explicit vulnerability language is easier to classify

The category-level analysis showed very strong performance for categories associated with explicit vulnerability terminology. The vulnerability category achieved an F1 score of 0.9973 across 368 observations.

In contrast, broader and more context-dependent categories such as authorization, cryptography, authentication, and secrets had lower F1 scores.

This suggests that the distinction between security-related and non-security-related comments becomes more difficult when security concepts are discussed as part of routine implementation, configuration, or development activities.

## 5. Model disagreement is relatively limited

The three models produced the same prediction for 1,806 of the 1,830 held-out test comments, corresponding to approximately 98.69% agreement.

Only 24 comments produced disagreement between at least two models.

The disagreement examples demonstrate that the models sometimes respond differently to contextual security language. These cases provide useful examples for qualitative error analysis rather than indicating that one model is universally correct.

# Final Model Interpretation

Based on the combined held-out test-set comparison and 5-fold stratified cross-validation results, Linear SVM provides the strongest overall observed performance among the three evaluated models for this dataset.

This interpretation is based on its higher mean accuracy, balanced accuracy, recall, and F1 score across the cross-validation folds, while Random Forest achieved slightly higher precision.

The results suggest that Linear SVM provides a useful balance between identifying security-related comments and limiting false positive classifications. However, the model still misses a number of security-related comments, particularly when the security context is indirect.

The final model should therefore be interpreted as a classifier for patterns associated with the constructed security-related labels rather than as a definitive vulnerability detector.

The results do not establish that the model can independently determine whether a software project contains an actual security vulnerability. Instead, they demonstrate that machine-learning models can learn distinguishable patterns in the constructed code review comment dataset and classify comments according to the labeling framework used in this study.

# Limitations

## 1. Labeling limitations

The final dataset contains labels generated through an LLM-assisted and rule-based labeling process. Although an independently reviewed validation dataset was available, no matching comment IDs were found between that validation dataset and the final labeled dataset. Therefore, direct agreement between the independent manual labels and the final labels could not be calculated.

The machine-learning results should consequently be interpreted as performance against the constructed labels rather than definitive measurements of real-world security-vulnerability detection.

## 2. Uncertain examples

The original labeling process included an uncertain class represented by label 2. These 83 examples were excluded from the binary classification experiments.

Excluding uncertain examples produces a clearer binary classification task, but it also means that ambiguous security-related comments are not represented in the reported binary performance metrics.

## 3. Dataset imbalance

The binary dataset contains more non-security-related comments than security-related comments. The class distribution is:

- Class 0: 6,119 comments
- Class 1: 3,028 comments

Class weighting and balanced accuracy were therefore considered during model evaluation.

## 4. Dataset construction

The candidate-generation process relies on security-related indicators and contextual rules. Consequently, comments that do not contain the selected indicators may not be represented in the final security candidate dataset.

This creates a potential selection limitation: the final dataset is not necessarily representative of every possible security-related code review comment.

## 5. Dataset size and source

The analysis is based on the available GHTorrent-derived pull-request comment dataset and its associated metadata. The dataset represents a particular collection and time period of GitHub activity and therefore may not represent all software projects, programming languages, development practices, or code-review platforms.

## 6. Small category sizes

Some security categories contain relatively few observations. For example, code execution, malicious activity, memory safety, and denial-of-service categories have small sample sizes.

Performance results for these categories should therefore be interpreted cautiously and should not be generalized as strongly as results from larger categories.

## 7. Text-only representation

The primary model representation uses TF-IDF features extracted from review comments. This means that the models primarily learn from textual patterns and do not directly analyze source-code structure, patches, data-flow relationships, or the actual vulnerability mechanics in the underlying software.

# Research Takeaway

The experiments demonstrate that traditional machine-learning approaches using TF-IDF representations can classify security-related code review comments with consistently high performance on the constructed dataset.

Across both the held-out test-set comparison and 5-fold stratified cross-validation, Linear SVM produced the strongest overall combination of accuracy, balanced accuracy, recall, and F1 score among the evaluated models, while Random Forest produced the highest precision.

The analysis also identified an important limitation of text-based classification: comments containing explicit vulnerability terminology are generally easier to classify than comments where security concerns are expressed indirectly through authentication, authorization, credentials, tokens, sessions, or cryptographic configuration.

These findings indicate that future improvements could focus on richer contextual representations and stronger labeling validation, particularly for ambiguous and context-dependent security discussions.